# quanteo - European Option Pricing & Greeks
**Author:** Matteo Ientile | **Environment:** Python 3.10+ (`quanteo` library)

This notebook demonstrates the pricing of European Options and Greeks calculation.

In [4]:
import numpy as np

# Quanteo Libraries
from quanteo.options import EuropeanOption, SumPricesCV, ArithmeticAsianOption, GeometricAsianOption
from quanteo.pricers import MonteCarloPricer, QuasiMCPricer, BSMPricer, ControlVariateMC, GeometricAsianPricer
from quanteo.models import GBM
from quanteo.risk import AnalyticalBSMGreeks, FiniteDifferenceGreek

In [5]:
#1.  ======================================= Market data and Contract structure
S0 = 100 #current price
K = 100 #strike price
ttm = 3/12 # option expires in 3 months
r = 0.03 # risk free rate
sigma = 0.45 #high volatility

# Define contract
contract = EuropeanOption(
    T=ttm, # pass directly time to maturity, then t_time = 0
    K=K,
    option_type="call"
)

In [6]:
#2.  ======================================= Define pricers and How to model the underlying asset
n_paths = 2**15
n_steps = 1 # we only care about the final price in this situation, there is no point to simulate multiple steps
SEED = 42
alpha=0.95

#underlying asset model
model = GBM(
    S0=S0,
    r=r,
    sigma=sigma,
    t_time=0.0
)

#crude Monte Carlo pricer
mc_pricer = MonteCarloPricer(
    n_paths=n_paths,
    n_steps=n_steps,
    seed=SEED,
    alpha=alpha,
    antithetic=False
)

#Monte Carlo with Antithetic Sampling (Variance Reduction method)
mc_anti_pricer = MonteCarloPricer(
    n_paths=n_paths,
    n_steps=n_steps,
    seed=SEED,
    alpha=alpha,
    antithetic=True
)

#Quasi-Monte Carlo pricer
qmc_pricer = QuasiMCPricer(
    n_paths=n_paths,
    n_steps=n_steps,
    seed=SEED
)

#Black-Scholes-Merton pricer (analytical solutions)
bsm_pricer = BSMPricer()

In [7]:
#3.  ======================================= Compute Option Price and CI
mc_price = mc_pricer.price(
    option=contract,
    model=model
)
ci_mc = list(mc_price.metrics.values())[0]

mc_anti_price = mc_anti_pricer.price(
    option=contract,
    model=model
)
ci_anti_mc = list(mc_anti_price.metrics.values())[0]

qmc_price = qmc_pricer.price(
    option=contract,
    model=model
)

bsm_price = bsm_pricer.price(
    option=contract,
    model=model
)

In [8]:
#4.  ======================================= Compute Greeks
bsm_risk = AnalyticalBSMGreeks()
fd_risk = FiniteDifferenceGreek(
    pricer=mc_anti_pricer, 
    dS_percentage=0.01,
    dr = 0.01,
    dttm = 1/365
) #use Monte Carlo with antithetic sampling as pricer

greeks_analytical = bsm_risk.calculate(
    option=contract,
    model=model
) 

greeks_numerical = fd_risk.greeks_calculator(
    option=contract,
    model=model
)

In [9]:
#5. ======================================= Print Results

title = "QUANT LIBRARY TEST - EUROPEAN CALL ATM (HIGH VOL REGIME)"
subtitle_price = "--- 1. PRICING ---"
subtitle_greeks = "--- 2. HEDGING (GREEKS) ---"

#title
print("="*100)
print(f"{title:^100}")
print("="*100)

print("")

#prices section
print(f"{subtitle_price:^100}")
print("")
print(f"{'Analytical (BSM) Price':<50} : {bsm_price.price:.4f}")
print(f"{'Crude MC Price':<50} : {mc_price.price:.4f} | {alpha*100}% CI : [{ci_mc[0]:.4f}, {ci_mc[1]:.4f}]")
print(f"{'Crude MC (with Antithetic Sampling) Price':<50} : {mc_anti_price.price:.4f} | {alpha*100}% CI : [{ci_anti_mc[0]:.4f}, {ci_anti_mc[1]:.4f}]")
print(f"{'Quasi MC Price':<50} : {qmc_price.price:.4f}")

print("")

#greeks section
print(f"{subtitle_greeks:^100}")
print("")
print(f"{'Analytical (BSM)':>25}  {'Finite Differences (MC with Antithetic Sampling & CRN)':>72}")
print(f"")
print(f"{'Delta':<7} : {greeks_analytical['Delta']:.4f} {greeks_numerical['Delta']:>45.4f}")
print(f"{'Gamma':<7} : {greeks_analytical['Gamma']:.4f} {greeks_numerical['Gamma']:>45.4f}")
print(f"{'Vega':<7} : {greeks_analytical['Vega']:.4f} {greeks_numerical['Vega']:>45.4f}")
print(f"{'Rho':<7} : {greeks_analytical['Rho']:.4f} {greeks_numerical['Rho']:>45.4f}")
print(f"{'Theta':<7} : {greeks_analytical['Theta']:.4f} {greeks_numerical['Theta']:>45.4f}")
print("="*100)


                      QUANT LIBRARY TEST - EUROPEAN CALL ATM (HIGH VOL REGIME)                      

                                         --- 1. PRICING ---                                         

Analytical (BSM) Price                             : 9.3024
Crude MC Price                                     : 9.4559 | 95.0% CI : [9.2884, 9.6235]
Crude MC (with Antithetic Sampling) Price          : 9.3139 | 95.0% CI : [9.1816, 9.4462]
Quasi MC Price                                     : 9.3021

                                    --- 2. HEDGING (GREEKS) ---                                     

         Analytical (BSM)                    Finite Differences (MC with Antithetic Sampling & CRN)

Delta   : 0.5580                                        0.5567
Gamma   : 0.0175                                        0.0178
Vega    : 19.7361                                       19.8004
Rho     : 11.6237                                       11.5897
Theta   : -19.1574                    